# **Paired Per-Query Tests Between Configurations**

This notebook reads the evaluation runs saved by the evaluation script (`runs/Eval/eval_<tag>/<method>_<student>/<dataset>.json`), computes per-query scores using `pytrec_eval` against the qrels loaded by `testbed.load`, and compares configurations query by query.

| Comparison family | A                                    | B                               |
| ----------------- | ------------------------------------ | ------------------------------- |
| `stepx4_vs_base`  | `eval_base_stepx4/<model>/<ds>`      | `eval_base/<model>/<ds>`        |
| `gpl_bm25_vs_gpl` | `eval_<tag>/gpl_bm25_<student>/<ds>` | `eval_<tag>/gpl_<student>/<ds>` |

`delta = A − B` (positive values favor A).

For each pair and metric, a single statistical test is applied: the **two-sided paired bootstrap test** (bootstrap-t on per-query differences, Sakai 2006).

A difference is considered statistically significant if `p < ALPHA`.


In [2]:
!pip install pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 109.8 MB/s eta 0:00:0000:01:01


In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import pytrec_eval

HERE = Path.cwd().resolve()         
PROJECT_ROOT = HERE.parents[1]

sys.path.insert(0, str(PROJECT_ROOT))

import testbed

ROOT = HERE
EVAL_ROOT = ROOT / "runs" / "Eval"     
STATS_ROOT = ROOT / "Stats"

## 1. Configuration

In [ ]:
DATASETS = ["scifact", "nfcorpus", "arguana"]
STUDENTS = ["BGE", "MiniLM", "GPL_distilbert"]
METHODS = ["gpl", "genq", "gpl_bm25"]
MODELS = [f"{method}_{student}" for method in METHODS for student in STUDENTS]

# stepx4 vs base: same model, two tags
STEPX4_TAG = "base_stepx4"
BASE_TAG = "base"

# gpl_bm25 vs gpl: same tag and student, compared inside each of these tags
GPL_TAGS = [
    "base",
    "base_neg10",
    "base_neg20",
    "base_neg30",
    "base_neg40",
    "base_stepx4",
]


#   "P@10": ("P", 10), "map@100": ("map_cut", 100), "mrr": ("recip_rank", None)
METRICS = {
    "ndcg@10": ("ndcg_cut", 10),
    "recall@100": ("recall", 100),
}

N_BOOTSTRAP = 100_000
ALPHA = 0.05
SEED = 42
IGNORE_IDENTICAL_IDS = True

## 2. Qrels and pytrec_eval Evaluators

One `RelevanceEvaluator` per dataset (the same evaluator used by BEIR), reused across all evaluation runs.

In [ ]:
MEASURES = {name: (f"{m}.{k}" if k is not None else m) for name, (m, k) in METRICS.items()}
RESULT_KEYS = {name: measure.replace(".", "_") for name, measure in MEASURES.items()}
CUTOFFS = {name: k for name, (_, k) in METRICS.items()}

QRELS = {}
EVALUATORS = {}
for dataset in DATASETS:
    _, _, qrels = testbed.load(dataset)
    QRELS[dataset] = {
        str(q): {str(d): int(r) for d, r in docs.items()} for q, docs in qrels.items()
    }
    EVALUATORS[dataset] = pytrec_eval.RelevanceEvaluator(QRELS[dataset], set(MEASURES.values()))
    print(f"{dataset}: {len(QRELS[dataset])} judged queries")

scifact: 300 judged queries
nfcorpus: 323 judged queries
arguana: 1406 judged queries


## 3. Loading Evaluation Runs and Per-Query Metrics

In [6]:
def load_run(path):
    loader = getattr(testbed, "load_run", None)
    if callable(loader):
        run = loader(path)
    else:
        with open(path, encoding="utf-8") as f:
            run = json.load(f)
    if isinstance(run, dict) and isinstance(run.get("run"), dict):
        run = run["run"]
    if not isinstance(run, dict) or not all(isinstance(v, dict) for v in run.values()):
        raise ValueError(f"{path}: expected a run shaped as {{query_id: {{doc_id: score}}}}")
    return {str(q): {str(d): float(s) for d, s in docs.items()} for q, docs in run.items()}


_scores_cache = {}
per_query_rows = []


def per_query_scores(tag, model, dataset):
    """Returns (qids, {metric: np.array}) for one run, or None if the run is missing."""
    key = (tag, model, dataset)
    if key in _scores_cache:
        return _scores_cache[key]

    path = EVAL_ROOT / f"eval_{tag}" / model / f"{dataset}.json"
    if not path.exists():
        print(f"  Missing run: {path}")
        _scores_cache[key] = None
        return None

    run = load_run(path)
    if IGNORE_IDENTICAL_IDS:
        run = {q: {d: s for d, s in docs.items() if d != q} for q, docs in run.items()}

    per_query = EVALUATORS[dataset].evaluate(run)

    # Every judged query, as in BEIR. pytrec_eval omits queries absent from the run: score them 0.
    qids = sorted(QRELS[dataset])
    scores = {
        name: np.array([per_query.get(q, {}).get(RESULT_KEYS[name], 0.0) for q in qids])
        for name in METRICS
    }

    missing = sum(q not in per_query for q in qids)
    if missing:
        print(f"  Warning: {missing}/{len(qids)} judged queries missing from {path} (scored as 0)")
    median_depth = float(np.median([len(run.get(q, {})) for q in qids]))
    for name, k in CUTOFFS.items():
        if k is not None and median_depth < k:
            print(f"  Warning: {path} keeps ~{median_depth:.0f} docs per query, fewer than k in {name}")

    for i, qid in enumerate(qids):
        row = {"tag": tag, "model": model, "dataset": dataset, "qid": qid}
        row.update({name: float(scores[name][i]) for name in METRICS})
        per_query_rows.append(row)

    _scores_cache[key] = (qids, scores)
    return _scores_cache[key]

## 4. Paired Bootstrap Test

Two-sided bootstrap-t test:

1. $d$ = per-query differences, $t_obs = mean(d) / (sd(d) / sqrt(n))$.
2. Center the differences: $w = d − mean(d)$, so that they satisfy the null hypothesis (zero mean).
3. Resample queries with replacement $B$ times and compute $t*$ for each bootstrap sample.
4. $p = ({|t*| ≥ |t_obs|} + 1) / (B + 1)$.

In [7]:
def bootstrap_test(diff, n_boot, rng):
    """Two-sided paired bootstrap-t test of H0: mean per-query difference = 0."""
    n = diff.size
    sd = diff.std(ddof=1)
    if sd == 0:
        return 1.0 if diff.mean() == 0 else 0.0
    t_obs = abs(diff.mean() / (sd / np.sqrt(n)))
    w = diff - diff.mean()                  # shift to satisfy H0
    batch = max(1, 5_000_000 // n)
    hits, done = 0, 0
    while done < n_boot:
        b = min(batch, n_boot - done)
        sample = w[rng.integers(0, n, size=(b, n))]
        sd_star = sample.std(axis=1, ddof=1)
        mean_star = sample.mean(axis=1)
        t_star = np.divide(np.abs(mean_star) * np.sqrt(n), sd_star,
                           out=np.zeros(b), where=sd_star > 0)
        hits += int(np.count_nonzero(t_star >= t_obs - 1e-12))
        done += b
    return (hits + 1) / (n_boot + 1)

## 5. Comparisons

In [8]:
def build_comparisons():
    comparisons = []
    for dataset in DATASETS:
        for model in MODELS:
            comparisons.append({
                "family": "stepx4_vs_base",
                "label": model,
                "dataset": dataset,
                "a": (STEPX4_TAG, model),
                "b": (BASE_TAG, model),
            })
    for tag in GPL_TAGS:
        for dataset in DATASETS:
            for student in STUDENTS:
                comparisons.append({
                    "family": "gpl_bm25_vs_gpl",
                    "label": f"{tag} | {student}",
                    "dataset": dataset,
                    "a": (tag, f"gpl_bm25_{student}"),
                    "b": (tag, f"gpl_{student}"),
                })
    return comparisons


rows = []
for comp in build_comparisons():
    print(f"[{comp['family']}] {comp['dataset']} | "
          f"{'/'.join(comp['a'])} vs {'/'.join(comp['b'])}")
    a = per_query_scores(*comp["a"], comp["dataset"])
    b = per_query_scores(*comp["b"], comp["dataset"])
    if a is None or b is None:
        print("  Skipped (run not found)")
        continue

    qids_a, scores_a = a
    qids_b, scores_b = b
    assert qids_a == qids_b, "Both runs must be scored on the same judged queries"

    for name in METRICS:
        x, y = scores_a[name], scores_b[name]
        diff = x - y
        rows.append({
            "family": comp["family"],
            "label": comp["label"],
            "dataset": comp["dataset"],
            "tag_a": comp["a"][0],
            "model_a": comp["a"][1],
            "tag_b": comp["b"][0],
            "model_b": comp["b"][1],
            "metric": name,
            "n_queries": int(diff.size),
            "mean_a": float(x.mean()),
            "mean_b": float(y.mean()),
            "delta": float(diff.mean()),
            "delta_rel_pct": float(100 * diff.mean() / y.mean()) if y.mean() > 0 else float("nan"),
            "wins": int(np.sum(diff > 0)),
            "ties": int(np.sum(diff == 0)),
            "losses": int(np.sum(diff < 0)),
            "p_boot": bootstrap_test(diff, N_BOOTSTRAP, np.random.default_rng(SEED)),
        })

print(f"\n{len(rows)} tests computed")

[stepx4_vs_base] scifact | base_stepx4/gpl_BGE vs base/gpl_BGE
[stepx4_vs_base] scifact | base_stepx4/gpl_MiniLM vs base/gpl_MiniLM
[stepx4_vs_base] scifact | base_stepx4/gpl_GPL_distilbert vs base/gpl_GPL_distilbert
[stepx4_vs_base] scifact | base_stepx4/genq_BGE vs base/genq_BGE
[stepx4_vs_base] scifact | base_stepx4/genq_MiniLM vs base/genq_MiniLM
[stepx4_vs_base] scifact | base_stepx4/genq_GPL_distilbert vs base/genq_GPL_distilbert
[stepx4_vs_base] scifact | base_stepx4/gpl_bm25_BGE vs base/gpl_bm25_BGE
[stepx4_vs_base] scifact | base_stepx4/gpl_bm25_MiniLM vs base/gpl_bm25_MiniLM
[stepx4_vs_base] scifact | base_stepx4/gpl_bm25_GPL_distilbert vs base/gpl_bm25_GPL_distilbert
[stepx4_vs_base] nfcorpus | base_stepx4/gpl_BGE vs base/gpl_BGE
[stepx4_vs_base] nfcorpus | base_stepx4/gpl_MiniLM vs base/gpl_MiniLM
[stepx4_vs_base] nfcorpus | base_stepx4/gpl_GPL_distilbert vs base/gpl_GPL_distilbert
[stepx4_vs_base] nfcorpus | base_stepx4/genq_BGE vs base/genq_BGE
[stepx4_vs_base] nfcorpus |

In [9]:
results = pd.DataFrame(rows)
results["significant"] = results["p_boot"] < ALPHA
results["mark"] = np.where(results["significant"], "*", "")

per_query = pd.DataFrame(per_query_rows)

STATS_ROOT.mkdir(parents=True, exist_ok=True)
results.to_csv(STATS_ROOT / "paired_tests.csv", index=False)
per_query.to_csv(STATS_ROOT / "per_query_metrics.csv", index=False)
print(f"Saved: {STATS_ROOT / 'paired_tests.csv'}")
print(f"Saved: {STATS_ROOT / 'per_query_metrics.csv'}")

Saved: /root/ir2_Project/pairs/10_GPL/runs/Stats/paired_tests.csv
Saved: /root/ir2_Project/pairs/10_GPL/runs/Stats/per_query_metrics.csv


## 6. Results

$`*` = p_{boot} < \alpha · W/T/L $ = queries won / tied / lost by A

In [10]:
NAMES = {"stepx4_vs_base": ("stepx4", "base"), "gpl_bm25_vs_gpl": ("gpl_bm25", "gpl")}

for (family, metric), table in results.groupby(["family", "metric"], sort=False):
    a_name, b_name = NAMES[family]
    n_sig = int(table["significant"].sum())
    n_up = int((table["significant"] & (table["delta"] > 0)).sum())
    print(f"\n{family} | {metric} | delta = {a_name} - {b_name} | {len(table)} tests | "
          f"significant: {n_sig} ({n_up} favour {a_name}, {n_sig - n_up} favour {b_name})")
    view = pd.DataFrame({
        "dataset": table["dataset"],
        "config": table["label"],
        a_name: table["mean_a"].round(4),
        b_name: table["mean_b"].round(4),
        "delta": table["delta"].map("{:+.4f}".format),
        "delta %": table["delta_rel_pct"].map("{:+.1f}%".format),
        "p_boot": table["p_boot"].round(4),
        "W/T/L": [f"{w}/{t}/{l}" for w, t, l in zip(table["wins"], table["ties"], table["losses"])],
        "": table["mark"],
    })
    display(view.reset_index(drop=True))


stepx4_vs_base | ndcg@10 | delta = stepx4 - base | 27 tests | significant: 16 (4 favour stepx4, 12 favour base)


,dataset,config,stepx4,base,delta,delta %,p_boot,W/T/L,
0,scifact,gpl_BGE,0.4458,0.5115,-0.0657,-12.8%,0.0000,22/194/84,*
1,scifact,gpl_MiniLM,0.4380,0.4888,-0.0508,-10.4%,0.0000,30/193/77,*
2,scifact,gpl_GPL_distilbert,0.3663,0.3842,-0.0179,-4.7%,0.1267,42/197/61,
3,scifact,genq_BGE,0.6687,0.6721,-0.0033,-0.5%,0.6247,29/240/31,
4,scifact,genq_MiniLM,0.6497,0.6406,+0.0090,+1.4%,0.1979,38/232/30,
5,scifact,genq_GPL_distilbert,0.6057,0.5959,+0.0099,+1.7%,0.3249,43/221/36,
6,scifact,gpl_bm25_BGE,0.4360,0.4719,-0.0358,-7.6%,0.0100,47/180/73,*
7,scifact,gpl_bm25_MiniLM,0.4675,0.4733,-0.0057,-1.2%,0.6052,42/206/52,
8,scifact,gpl_bm25_GPL_distilbert,0.3798,0.3928,-0.0130,-3.3%,0.3503,46/200/54,
9,nfcorpus,gpl_BGE,0.2891,0.3025,-0.0134,-4.4%,0.0129,57/167/99,*



stepx4_vs_base | recall@100 | delta = stepx4 - base | 27 tests | significant: 14 (2 favour stepx4, 12 favour base)


,dataset,config,stepx4,base,delta,delta %,p_boot,W/T/L,
0,scifact,gpl_BGE,0.8064,0.8589,-0.0524,-6.1%,0.0020,0/281/19,*
1,scifact,gpl_MiniLM,0.8206,0.8732,-0.0527,-6.0%,0.0015,3/278/19,*
2,scifact,gpl_GPL_distilbert,0.7281,0.7937,-0.0656,-8.3%,0.0007,8/264/28,*
3,scifact,genq_BGE,0.9433,0.9383,+0.0050,+0.5%,0.4128,3/296/1,
4,scifact,genq_MiniLM,0.9377,0.9293,+0.0083,+0.9%,0.0847,3/297/0,
5,scifact,genq_GPL_distilbert,0.8942,0.8937,+0.0006,+0.1%,0.9574,5/290/5,
6,scifact,gpl_bm25_BGE,0.7711,0.7631,+0.0080,+1.0%,0.6573,17/268/15,
7,scifact,gpl_bm25_MiniLM,0.8037,0.8131,-0.0094,-1.2%,0.4957,9/278/13,
8,scifact,gpl_bm25_GPL_distilbert,0.7152,0.7292,-0.0141,-1.9%,0.3782,11/272/17,
9,nfcorpus,gpl_BGE,0.2633,0.2753,-0.0120,-4.4%,0.0962,55/169/99,



gpl_bm25_vs_gpl | ndcg@10 | delta = gpl_bm25 - gpl | 54 tests | significant: 22 (15 favour gpl_bm25, 7 favour gpl)


,dataset,config,gpl_bm25,gpl,delta,delta %,p_boot,W/T/L,
0,scifact,base | BGE,0.4719,0.5115,-0.0396,-7.7%,0.0015,41/189/70,*
1,scifact,base | MiniLM,0.4733,0.4888,-0.0155,-3.2%,0.1778,46/200/54,
2,scifact,base | GPL_distilbert,0.3928,0.3842,+0.0086,+2.2%,0.5119,51/201/48,
3,nfcorpus,base | BGE,0.3094,0.3025,+0.0069,+2.3%,0.1211,85/175/63,
4,nfcorpus,base | MiniLM,0.2889,0.2873,+0.0016,+0.5%,0.6428,79/173/71,
5,nfcorpus,base | GPL_distilbert,0.2449,0.2435,+0.0013,+0.5%,0.7875,72/184/67,
6,arguana,base | BGE,0.3703,0.3594,+0.0109,+3.0%,0.0629,339/786/281,
7,arguana,base | MiniLM,0.4080,0.3708,+0.0373,+10.0%,0.0000,379/784/243,*
8,arguana,base | GPL_distilbert,0.3628,0.3295,+0.0333,+10.1%,0.0000,389/772/245,*
9,scifact,base_neg10 | BGE,0.4660,0.4979,-0.0319,-6.4%,0.0106,38/203/59,*



gpl_bm25_vs_gpl | recall@100 | delta = gpl_bm25 - gpl | 54 tests | significant: 32 (14 favour gpl_bm25, 18 favour gpl)


,dataset,config,gpl_bm25,gpl,delta,delta %,p_boot,W/T/L,
0,scifact,base | BGE,0.7631,0.8589,-0.0958,-11.2%,0.0000,1/266/33,*
1,scifact,base | MiniLM,0.8131,0.8732,-0.0601,-6.9%,0.0008,4/272/24,*
2,scifact,base | GPL_distilbert,0.7292,0.7937,-0.0644,-8.1%,0.0011,9/262/29,*
3,nfcorpus,base | BGE,0.2693,0.2753,-0.0060,-2.2%,0.3527,64/173/86,
4,nfcorpus,base | MiniLM,0.2704,0.2659,+0.0046,+1.7%,0.2722,71/186/66,
5,nfcorpus,base | GPL_distilbert,0.2107,0.2099,+0.0008,+0.4%,0.9013,67/188/68,
6,arguana,base | BGE,0.9075,0.8997,+0.0078,+0.9%,0.1567,36/1345/25,
7,arguana,base | MiniLM,0.9538,0.9225,+0.0313,+3.4%,0.0000,57/1336/13,*
8,arguana,base | GPL_distilbert,0.9011,0.8890,+0.0121,+1.4%,0.0720,53/1317/36,
9,scifact,base_neg10 | BGE,0.7419,0.8426,-0.1007,-11.9%,0.0000,4/261/35,*


### Resumen compacto: delta por configuración × dataset

In [11]:
for (family, metric), table in results.groupby(["family", "metric"], sort=False):
    print(f"\n{family} | {metric}")
    cell = table["delta"].map("{:+.4f}".format) + table["mark"]
    pivot = (table.assign(cell=cell)
                  .pivot(index="label", columns="dataset", values="cell")
                  .reindex(columns=[d for d in DATASETS if d in set(table["dataset"])])
                  .reindex(index=list(dict.fromkeys(table["label"]))))
    display(pivot.fillna("—"))


stepx4_vs_base | ndcg@10


dataset,scifact,nfcorpus,arguana
label,,,
gpl_BGE,-0.0657*,-0.0134*,-0.0186*
gpl_MiniLM,-0.0508*,-0.0205*,-0.0143*
gpl_GPL_distilbert,-0.0179,-0.0036,-0.0047
genq_BGE,-0.0033,+0.0002,+0.0236*
genq_MiniLM,+0.0090,+0.0034,+0.0090*
genq_GPL_distilbert,+0.0099,+0.0094*,+0.0390*
gpl_bm25_BGE,-0.0358*,-0.0186*,-0.0282*
gpl_bm25_MiniLM,-0.0057,-0.0149*,-0.0500*
gpl_bm25_GPL_distilbert,-0.0130,-0.0067,-0.0196*



stepx4_vs_base | recall@100


dataset,scifact,nfcorpus,arguana
label,,,
gpl_BGE,-0.0524*,-0.0120,-0.0171*
gpl_MiniLM,-0.0527*,-0.0232*,-0.0156*
gpl_GPL_distilbert,-0.0656*,-0.0006,-0.0284*
genq_BGE,+0.0050,-0.0078,-0.0007
genq_MiniLM,+0.0083,-0.0019,+0.0014
genq_GPL_distilbert,+0.0006,+0.0095*,+0.0085*
gpl_bm25_BGE,+0.0080,-0.0260*,-0.0320*
gpl_bm25_MiniLM,-0.0094,-0.0306*,-0.0341*
gpl_bm25_GPL_distilbert,-0.0141,-0.0108,-0.0128*



gpl_bm25_vs_gpl | ndcg@10


dataset,scifact,nfcorpus,arguana
label,,,
base | BGE,-0.0396*,+0.0069,+0.0109
base | MiniLM,-0.0155,+0.0016,+0.0373*
base | GPL_distilbert,+0.0086,+0.0013,+0.0333*
base_neg10 | BGE,-0.0319*,+0.0054,+0.0120*
base_neg10 | MiniLM,-0.0537*,-0.0002,+0.0265*
base_neg10 | GPL_distilbert,-0.0027,+0.0053,+0.0258*
base_neg20 | BGE,-0.0489*,+0.0024,+0.0239*
base_neg20 | MiniLM,-0.0399*,-0.0008,+0.0395*
base_neg20 | GPL_distilbert,-0.0133,+0.0018,+0.0366*



gpl_bm25_vs_gpl | recall@100


dataset,scifact,nfcorpus,arguana
label,,,
base | BGE,-0.0958*,-0.0060,+0.0078
base | MiniLM,-0.0601*,+0.0046,+0.0313*
base | GPL_distilbert,-0.0644*,+0.0008,+0.0121
base_neg10 | BGE,-0.1007*,-0.0111,+0.0284*
base_neg10 | MiniLM,-0.0763*,-0.0140*,+0.0391*
base_neg10 | GPL_distilbert,-0.0614*,+0.0039,+0.0519*
base_neg20 | BGE,-0.1047*,-0.0148*,+0.0306*
base_neg20 | MiniLM,-0.0668*,-0.0025,+0.0356*
base_neg20 | GPL_distilbert,-0.0772*,-0.0017,+0.0519*
